# Run `vecAdd.cu` in Google Colab
Use a GPU runtime in Colab, then run the two cells below.

In [1]:
%%writefile vecAdd.cu
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <cstdio>
#include <vector>

#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            std::fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__,         \
                         __LINE__, cudaGetErrorString(err));                    \
            std::exit(EXIT_FAILURE);                                            \
        }                                                                       \
    } while (0)

void vecAddH(const float *A_h, const float *B_h, float *C_h, int n) {
    for (int i = 0; i < n; i++) {
        C_h[i] = A_h[i] + B_h[i];
    }
}

__global__ void vecAddKernel(const float *A, const float *B, float *C, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

void vecAddD(const float *A, const float *B, float *C, int n) {
    int size = n * static_cast<int>(sizeof(float));
    float *A_d = nullptr;
    float *B_d = nullptr;
    float *C_d = nullptr;

    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&A_d), size));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&B_d), size));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&C_d), size));

    CUDA_CHECK(cudaMemcpy(A_d, A, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B_d, B, size, cudaMemcpyHostToDevice));

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;
    vecAddKernel<<<blocksPerGrid, threadsPerBlock>>>(A_d, B_d, C_d, n);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(C, C_d, size, cudaMemcpyDeviceToHost));

    CUDA_CHECK(cudaFree(A_d));
    CUDA_CHECK(cudaFree(B_d));
    CUDA_CHECK(cudaFree(C_d));
}

double timeVecAddH(const float *A, const float *B, float *C, int n, int runs) {
    auto start = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < runs; i++) {
        vecAddH(A, B, C, n);
    }
    auto end = std::chrono::high_resolution_clock::now();
    double totalMs =
        std::chrono::duration<double, std::milli>(end - start).count();
    return totalMs / static_cast<double>(runs);
}

double timeVecAddDEndToEnd(const float *A, const float *B, float *C, int n,
                           int runs) {
    auto start = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < runs; i++) {
        vecAddD(A, B, C, n);
    }
    auto end = std::chrono::high_resolution_clock::now();
    double totalMs =
        std::chrono::duration<double, std::milli>(end - start).count();
    return totalMs / static_cast<double>(runs);
}

double timeVecAddDKernelOnly(const float *A, const float *B, float *C, int n,
                             int runs) {
    int size = n * static_cast<int>(sizeof(float));
    float *A_d = nullptr;
    float *B_d = nullptr;
    float *C_d = nullptr;
    cudaEvent_t startEvent;
    cudaEvent_t stopEvent;

    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&A_d), size));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&B_d), size));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&C_d), size));
    CUDA_CHECK(cudaMemcpy(A_d, A, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(B_d, B, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventCreate(&startEvent));
    CUDA_CHECK(cudaEventCreate(&stopEvent));

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    // Warm up the kernel once before collecting timing.
    vecAddKernel<<<blocksPerGrid, threadsPerBlock>>>(A_d, B_d, C_d, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    float totalKernelMs = 0.0f;
    for (int i = 0; i < runs; i++) {
        CUDA_CHECK(cudaEventRecord(startEvent));
        vecAddKernel<<<blocksPerGrid, threadsPerBlock>>>(A_d, B_d, C_d, n);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(stopEvent));
        CUDA_CHECK(cudaEventSynchronize(stopEvent));

        float iterMs = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&iterMs, startEvent, stopEvent));
        totalKernelMs += iterMs;
    }

    CUDA_CHECK(cudaMemcpy(C, C_d, size, cudaMemcpyDeviceToHost));

    CUDA_CHECK(cudaEventDestroy(startEvent));
    CUDA_CHECK(cudaEventDestroy(stopEvent));
    CUDA_CHECK(cudaFree(A_d));
    CUDA_CHECK(cudaFree(B_d));
    CUDA_CHECK(cudaFree(C_d));

    return static_cast<double>(totalKernelMs) / static_cast<double>(runs);
}

int main() {
    const int n = 10000;
    const int cpuRuns = 5000;
    const int gpuEndToEndRuns = 100;
    const int gpuKernelRuns = 1000;
    std::vector<float> A(n), B(n), C_h(n), C_d(n);

    for (int i = 0; i < n; i++) {
        A[i] = static_cast<float>(i) * 0.5f;
        B[i] = static_cast<float>(i) * 1.5f;
    }

    // Warm up CUDA context and a single end-to-end run before benchmarking.
    CUDA_CHECK(cudaFree(0));
    vecAddD(A.data(), B.data(), C_d.data(), n);

    double cpuMs = timeVecAddH(A.data(), B.data(), C_h.data(), n, cpuRuns);
    double gpuEndToEndMs = timeVecAddDEndToEnd(A.data(), B.data(), C_d.data(),
                                               n, gpuEndToEndRuns);
    double gpuKernelMs =
        timeVecAddDKernelOnly(A.data(), B.data(), C_d.data(), n, gpuKernelRuns);

    float maxAbsErr = 0.0f;
    for (int i = 0; i < n; i++) {
        float err = std::fabs(C_h[i] - C_d[i]);
        if (err > maxAbsErr) {
            maxAbsErr = err;
        }
    }

    std::printf("Vector size: %d\n", n);
    std::printf("CPU vecAddH avg time (%d runs): %.6f ms\n", cpuRuns, cpuMs);
    std::printf(
        "GPU vecAddD end-to-end avg time (%d runs, malloc+H2D+kernel+D2H+free): "
        "%.6f ms\n",
        gpuEndToEndRuns, gpuEndToEndMs);
    std::printf("GPU kernel-only avg time (%d runs): %.6f ms\n", gpuKernelRuns,
                gpuKernelMs);
    std::printf("Max absolute error: %.6f\n", maxAbsErr);
    if (gpuEndToEndMs > 0.0) {
        std::printf("Speedup (CPU/GPU end-to-end): %.4fx\n",
                    cpuMs / gpuEndToEndMs);
    }
    if (gpuKernelMs > 0.0) {
        std::printf("Speedup (CPU/GPU kernel-only): %.4fx\n",
                    cpuMs / gpuKernelMs);
    }

    return 0;
}


Overwriting vecAdd.cu


In [ ]:
!nvcc -std=c++14 vecAdd.cu -o vecAdd
!./vecAdd
